# Overview
This notebook validates the analytical dataset for Power BI by assessing data grain, identifying aggregation risks, and defining the semantic data model required to build accurate, reliable, and interactive business dashboards.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np

from config.paths import ANALYTICS_SALES_FILE

from utils.file_utils import read_csv

In [3]:
sales_df = read_csv(ANALYTICS_SALES_FILE)

sales_df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,InvoiceYear,...,MonthName,Quarter,MonthStart,YearMonth,CustomerType,PurchaseFrequency,RepeatCustomer,OrderType,InvoiceRevenue,BasketSize
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,139.12,7
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,139.12,7
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,139.12,7
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,139.12,7
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,139.12,7
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,15.30,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,139.12,7
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,25.50,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,139.12,7
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,22.20,2
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,35.0,Repeat Customer,Sale,22.20,2
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,54.08,2010,...,December,Q4,2010-12-01,Dec-2010,Registered Customer,18.0,Repeat Customer,Sale,278.73,12


In [4]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534593 entries, 0 to 534592
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   InvoiceNo          534593 non-null  object 
 1   StockCode          534593 non-null  object 
 2   Description        534593 non-null  object 
 3   Quantity           534593 non-null  int64  
 4   InvoiceDate        534593 non-null  object 
 5   UnitPrice          534593 non-null  float64
 6   CustomerID         401598 non-null  float64
 7   Country            534593 non-null  object 
 8   Revenue            534593 non-null  float64
 9   InvoiceYear        534593 non-null  int64  
 10  InvoiceMonth       534593 non-null  int64  
 11  MonthName          534593 non-null  object 
 12  Quarter            534593 non-null  object 
 13  MonthStart         534593 non-null  object 
 14  YearMonth          534593 non-null  object 
 15  CustomerType       534593 non-null  object 
 16  Pu

In [5]:
sales_df["InvoiceDate"] = pd.to_datetime(
    sales_df["InvoiceDate"]
)

In [6]:
sales_df["MonthStart"] = pd.to_datetime(
    sales_df["MonthStart"]
)

In [7]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534593 entries, 0 to 534592
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   InvoiceNo          534593 non-null  object        
 1   StockCode          534593 non-null  object        
 2   Description        534593 non-null  object        
 3   Quantity           534593 non-null  int64         
 4   InvoiceDate        534593 non-null  datetime64[ns]
 5   UnitPrice          534593 non-null  float64       
 6   CustomerID         401598 non-null  float64       
 7   Country            534593 non-null  object        
 8   Revenue            534593 non-null  float64       
 9   InvoiceYear        534593 non-null  int64         
 10  InvoiceMonth       534593 non-null  int64         
 11  MonthName          534593 non-null  object        
 12  Quarter            534593 non-null  object        
 13  MonthStart         534593 non-null  datetime

### Grain validation

In [8]:
# Dataset Grain Validation
grain_summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Distinct Invoices",
        "Distinct Customers",
        "Distinct Products"
    ],
    "Value": [
        len(sales_df),
        sales_df["InvoiceNo"].nunique(),
        sales_df["CustomerID"].nunique(),
        sales_df["StockCode"].nunique()
    ]
})

grain_summary

,Metric,Value
0,Total Rows,534593
1,Distinct Invoices,23857
2,Distinct Customers,4372
3,Distinct Products,3940


In [9]:
sample_invoice = sales_df["InvoiceNo"].iloc[0]

sales_df.loc[
    sales_df["InvoiceNo"] == "536365",
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "Revenue"
    ]
]

,InvoiceNo,StockCode,Description,Quantity,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,15.30
1,536365,71053,WHITE METAL LANTERN,6,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,20.34
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,15.30
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,25.50


**Nonte: One row represents one product sold within a single invoice (Invoice Line Grain).**

In [10]:
# Invoice-Level Validation

invoice_validation = (
    sales_df
    .groupby("InvoiceNo")
    .agg(
        InvoiceRevenue_Unique=("InvoiceRevenue", "nunique"),
        BasketSize_Unique=("BasketSize", "nunique"),
        OrderType_Unique=("OrderType", "nunique")
    )
)

invoice_validation.head()

,InvoiceRevenue_Unique,BasketSize_Unique,OrderType_Unique
InvoiceNo,,,
536365,1,1,1
536366,1,1,1
536367,1,1,1
536368,1,1,1
536369,1,1,1


In [11]:
# Validation Summary

validation_summary = pd.DataFrame({
    "Validation": [
        "InvoiceRevenue constant per invoice",
        "BasketSize constant per invoice",
        "OrderType constant per invoice"
    ],
    "Status": [
        "✅" if (invoice_validation["InvoiceRevenue_Unique"] == 1).all() else "❌",
        "✅" if (invoice_validation["BasketSize_Unique"] == 1).all() else "❌",
        "✅" if (invoice_validation["OrderType_Unique"] == 1).all() else "❌"
    ]
})

validation_summary

,Validation,Status
0,InvoiceRevenue constant per invoice,✅
1,BasketSize constant per invoice,✅
2,OrderType constant per invoice,✅


All invoice-level attributes remain consistent across product lines, confirming that InvoiceRevenue, BasketSize, and OrderType belong to the invoice grain. These fields can therefore be separated into a dedicated FactInvoice table without introducing data quality or aggregation issues.

In [12]:
# Customer-Level Validation

customer_validation = (
    sales_df.loc[sales_df["CustomerID"].notna()]
    .groupby("CustomerID")
    .agg(
        PurchaseFrequency_Unique=("PurchaseFrequency", "nunique"),
        RepeatCustomer_Unique=("RepeatCustomer", "nunique"),
        CustomerType_Unique=("CustomerType", "nunique")
    )
)

customer_validation.head()

,PurchaseFrequency_Unique,RepeatCustomer_Unique,CustomerType_Unique
CustomerID,,,
12346.0,1,1,1
12347.0,1,1,1
12348.0,1,1,1
12349.0,1,1,1
12350.0,1,1,1


In [13]:
# Validation Summary

customer_validation_summary = pd.DataFrame({
    "Validation": [
        "PurchaseFrequency constant per customer",
        "RepeatCustomer constant per customer",
        "CustomerType constant per customer"
    ],
    "Status": [
        "✅" if (customer_validation["PurchaseFrequency_Unique"] == 1).all() else "❌",
        "✅" if (customer_validation["RepeatCustomer_Unique"] == 1).all() else "❌",
        "✅" if (customer_validation["CustomerType_Unique"] == 1).all() else "❌"
    ]
})

customer_validation_summary

,Validation,Status
0,PurchaseFrequency constant per customer,✅
1,RepeatCustomer constant per customer,✅
2,CustomerType constant per customer,✅


Customer-level attributes remain consistent for every customer, confirming that PurchaseFrequency, RepeatCustomer, and CustomerType belong to the customer grain. These fields can therefore be separated into a dedicated DimCustomer table, enabling accurate customer-level analysis and preventing aggregation issues in Power BI.

In [14]:
# One Description per StockCode


stockcode_validation = (
    sales_df
    .groupby("StockCode")["Description"]
    .nunique()
)

stockcode_validation.value_counts().sort_index()

Description
1    3680
2     237
3      17
4       5
5       1
Name: count, dtype: int64

In [15]:
# StockCodes with multiple descriptions

invalid_stockcodes = stockcode_validation[stockcode_validation > 1].index

sales_df.loc[
    sales_df["StockCode"].isin(invalid_stockcodes),
    ["StockCode", "Description"]
].sort_values("StockCode")

,StockCode,Description
273753,16156L,WRAP CAROUSEL
101915,16156L,"WRAP, CAROUSEL"
70975,16156L,"WRAP, CAROUSEL"
190585,16156L,WRAP CAROUSEL
328004,16156L,WRAP CAROUSEL
...,...,...
90106,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00
231134,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00
44131,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00
159221,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00


In [16]:
# Standardize Product Descriptions

sales_df["Description"] = (
    sales_df["Description"]
    .str.strip()                  # Remove leading/trailing spaces
    .str.replace(",", "", regex=False)  # Remove commas
    .str.replace(r"\s+", " ", regex=True)  # Replace multiple spaces with one
)

In [17]:
stockcode_validation = (
    sales_df
    .groupby("StockCode")["Description"]
    .nunique()
)

stockcode_validation.value_counts().sort_index()

Description
1    3700
2     218
3      16
4       5
5       1
Name: count, dtype: int64

Product validation identified minor description inconsistencies for a small proportion of product codes, primarily caused by formatting differences rather than distinct products. As the analytical dataset contains only basic product attributes (StockCode and Description), a dedicated DimProduct table would provide limited analytical value for the current reporting requirements. Therefore, product information will remain within the sales fact table.

In [18]:
# ---------------------------------------
# Date Range Validation
# ---------------------------------------

date_summary = pd.DataFrame({
    "Metric": [
        "Start Date",
        "End Date",
        "Total Days"
    ],
    "Value": [
        sales_df["InvoiceDate"].min().date(),
        sales_df["InvoiceDate"].max().date(),
        (
            sales_df["InvoiceDate"].max()
            - sales_df["InvoiceDate"].min()
        ).days + 1
    ]
})

date_summary

,Metric,Value
0,Start Date,2010-12-01
1,End Date,2011-12-09
2,Total Days,374


In [21]:
# Monthly Coverage Validation

monthly_coverage = (
    sales_df
    .groupby("MonthStart")
    .size()
    .reset_index(name="Transactions")
    .sort_values("MonthStart")
)

monthly_coverage

,MonthStart,Transactions
0,2010-12-01,41842
1,2011-01-01,34770
2,2011-02-01,27406
3,2011-03-01,36231
4,2011-04-01,29447
5,2011-05-01,36622
6,2011-06-01,36462
7,2011-07-01,39090
8,2011-08-01,34967
9,2011-09-01,49715


In [22]:
expected_months = pd.date_range(
    start=monthly_coverage["MonthStart"].min(),
    end=monthly_coverage["MonthStart"].max(),
    freq="MS"
)

missing_months = expected_months.difference(
    monthly_coverage["MonthStart"]
)

missing_months

DatetimeIndex([], dtype='datetime64[ns]', freq='MS')

In [23]:
# Continuous Timeline Validation

continuous_timeline = len(missing_months) == 0

continuous_timeline

True

In [24]:
# ---------------------------------------
# Validation Summary
# ---------------------------------------

date_validation_summary = pd.DataFrame({
    "Validation": [
        "Continuous timeline",
        "Missing months",
        "Calendar coverage"
    ],
    "Status": [
        "✅" if continuous_timeline else "❌",
        "✅" if len(missing_months) == 0 else "❌",
        "✅"
    ]
})

date_validation_summary

,Validation,Status
0,Continuous timeline,✅
1,Missing months,✅
2,Calendar coverage,✅


The analytical dataset provides complete and continuous calendar coverage across the reporting period. A dedicated DimDate table should be created to support hierarchical filtering, time intelligence, and consistent date relationships within the Power BI semantic model.

# Relationship Assessment

| Table Name         | Primary Key | Foreign Key | Data Grain            | Business Purpose                                                                                      |
| ------------------ | ----------- | ----------- | --------------------- | ----------------------------------------------------------------------------------------------------- |
| **SalesAnalytics** | —           | —           | **Invoice × Product** | Stores transaction-level sales records used for revenue, product, customer, and geographic analysis. |
| **DimCustomer**    | CustomerID  | CustomerID  | **Customer**          | Stores unique customer attributes to support customer segmentation and purchasing behavior analysis. |
| **DimDate**        | Date        | InvoiceDate | **Date**              | Supports time intelligence, trend analysis, and calendar-based reporting.                             |                                     |

# Model Relationships

| From Table   | Relationship  | To Table        | Join Column               | Cardinality |
| ------------ | ------------- | --------------- | ------------------------- | ----------- |
| DimCustomer  | One-to-Many   | SalesAnalytics  | CustomerID                | 1 : *       |
| DimDate      | One-to-Many   | SalesAnalytics  | Date → InvoiceDate       | 1 : *       |

# Model Summary

| Model Type      | Fact Tables      | Dimension Tables     | Primary Analysis Grain                 |
| --------------- | ---------------- | -------------------- | -------------------------------------- |
| **Star Schema** | SalesAnalytics   | DimCustomer, DimDate | **Invoice Line (Product Transaction)** |